In [1]:
# 01_train_valid_test_split
#
# 목적: train.csv(2월 만료 코호트) 기준으로 유저(msno) 그룹 stratified 분할로
#      train/valid/test(70/15/15)를 만든다.
#
#      피처는 이 코호트의 예측 시점인 2017-01-31까지의 데이터만 사용한다.
#
# 입력: data/raw/train.csv
# 출력:
#   data/processed/labels_pooled.csv   (msno, snapshot, is_churn) - 라벨 테이블
#                                        (snapshot은 피처 관측 기준일인 "2017-01-31")
#   data/processed/user_split.csv      (msno, split)              - 유저별 split 배정

In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RAW_DIR = Path("../data/raw").resolve()
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.30    # temp = valid + test
VALID_RATIO_OF_TEMP = 0.50  # temp를 valid/test로 반씩

In [3]:
train = pd.read_csv(RAW_DIR / "train.csv")
train["snapshot"] = "2017-01-31"

pooled = train[["msno", "snapshot", "is_churn"]].copy()
print(f"라벨 행 수: {len(pooled):,}")
pooled.head()

라벨 행 수: 992,931


,msno,snapshot,is_churn
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,2017-01-31,1
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,2017-01-31,1
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,2017-01-31,1
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,2017-01-31,1
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,2017-01-31,1


In [4]:
# 유저별 대표 라벨 (이제 유저당 스냅샷이 하나뿐이라 is_churn과 동일하지만,
# 이후 코드가 msno 인덱스 Series를 기대하므로 동일한 형태로 유지)
user_label = pooled.groupby("msno")["is_churn"].max()
print(f"고유 유저 수: {len(user_label):,}")
print(f"이탈 유저 비율: {user_label.mean() * 100:.2f}%")

고유 유저 수: 992,931
이탈 유저 비율: 6.39%


In [5]:
train_users, temp_users = train_test_split(
    user_label.index.to_numpy(), test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=user_label.values
)
temp_labels = user_label.loc[temp_users]
valid_users, test_users = train_test_split(
    temp_users, test_size=VALID_RATIO_OF_TEMP, random_state=RANDOM_STATE, stratify=temp_labels.values
)

train_users, valid_users, test_users = set(train_users), set(valid_users), set(test_users)

assert len(train_users & valid_users) == 0
assert len(train_users & test_users) == 0
assert len(valid_users & test_users) == 0
print("그룹 무결성 검증 통과: train/valid/test 간 유저 중복 없음")

그룹 무결성 검증 통과: train/valid/test 간 유저 중복 없음


In [6]:
split_map = {}
for u in train_users:
    split_map[u] = "train"
for u in valid_users:
    split_map[u] = "valid"
for u in test_users:
    split_map[u] = "test"

user_split = pd.Series(split_map, name="split").rename_axis("msno").reset_index()
pooled_with_split = pooled.merge(user_split, on="msno", how="left")

assert pooled_with_split["split"].isna().sum() == 0, "split이 배정되지 않은 행 존재"

summary = pooled_with_split.groupby("split").agg(
    rows=("msno", "size"), users=("msno", "nunique"), churn_rate=("is_churn", "mean")
)
summary

,rows,users,churn_rate
split,,,
test,148940,148940,0.063918
train,695051,695051,0.063923
valid,148940,148940,0.063925


In [7]:
pooled.to_csv(PROCESSED_DIR / "labels_pooled.csv", index=False)
user_split.to_csv(PROCESSED_DIR / "user_split.csv", index=False)

print(f"저장 완료: {PROCESSED_DIR / 'labels_pooled.csv'} ({len(pooled):,} rows)")
print(f"저장 완료: {PROCESSED_DIR / 'user_split.csv'} ({len(user_split):,} rows)")

저장 완료: ..\data\processed\labels_pooled.csv (992,931 rows)
저장 완료: ..\data\processed\user_split.csv (992,931 rows)
